In [7]:
import numpy as np
import pandas as pd
df = pd.read_csv("C:/Users/inter/OneDrive/Documents/GENAI/Skincare-ChatBot/ingredientsList.csv")

In [8]:
df.shape

(248, 8)

In [9]:
df.isnull().sum()

name                    1
scientific_name       247
short_description       1
what_is_it              1
what_does_it_do         1
who_is_it_good_for      0
who_should_avoid        0
url                     0
dtype: int64

In [10]:
df = df.drop('scientific_name', axis=1)

In [11]:
df.dropna(inplace=True)
df.isnull().sum()

name                  0
short_description     0
what_is_it            0
what_does_it_do       0
who_is_it_good_for    0
who_should_avoid      0
url                   0
dtype: int64

In [12]:
print(df.columns)
print(df.head())

Index(['name', 'short_description', 'what_is_it', 'what_does_it_do',
       'who_is_it_good_for', 'who_should_avoid', 'url'],
      dtype='str')
                           name  \
0  Alpha-Glucan Oligosaccharide   
1                     Aloe Vera   
2                     Allantoin   
3                         Algin   
4                 Algae Extract   

                                   short_description  \
0  Alpha-glucan oligosaccharide is in a class of ...   
1  Aloe vera, also appear on ingredients lists as...   
2  Allantoin occurs naturally in the body, but ca...   
3  Algin, also known as sodium alginate, is a lar...   
4  It is essentially an underwater plant, designe...   

                                          what_is_it  \
0  Prebiotics are a type of non-digestible dietar...   
1  Aloe vera is a skincare ingredient derived fro...   
2  Allantoin is a skincare ingredient derived fro...   
3  An extract from brown seaweed used as hydratin...   
4  An incredibly interestin

In [13]:
df["search_text"] = (
    df["name"] + ". " +
    df["short_description"] + " ." +
    df["what_does_it_do"] + ". " + df['who_is_it_good_for'] + " ." + df['who_should_avoid']
)

In [14]:
df['search_text'].head()

0    Alpha-Glucan Oligosaccharide. Alpha-glucan oli...
1    Aloe Vera. Aloe vera, also appear on ingredien...
2    Allantoin. Allantoin occurs naturally in the b...
3    Algin. Algin, also known as sodium alginate, i...
4    Algae Extract. It is essentially an underwater...
Name: search_text, dtype: str

In [16]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')
# Ensure all inputs are strings and keep alignment with the dataframe
texts = df['search_text'].fillna("").astype(str).tolist()
# Encode in batches and return a NumPy array
embeddings = model.encode(texts, show_progress_bar=True, batch_size=64, convert_to_numpy=True)
np.save("Ingredients_embedding.npy", embeddings)
df.to_pickle("ingredient_data.pkl")

Batches: 100%|██████████| 4/4 [00:12<00:00,  3.10s/it]


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def retrieve(query , model ,embedding,  df , top_k=5 , threshold=0.35):
    query_vec = model.encode([query])
    sims = cosine_similarity(query_vec , embedding)[0]
    top_idx = sims.argsort()[::-1][:top_k]
    max_sim = sims[top_idx[0]]

    if max_sim < threshold:
        return None, max_sim  

    results = df.iloc[top_idx].copy()
    results["similarity"] = sims[top_idx]
    return results, max_sim

In [24]:
from groq import Groq
from dotenv import load_dotenv
import os

load_dotenv()

print(os.getenv("GROQ_API_KEY"))

client = Groq()

SYSTEM_PROMPT = """You are a skincare recommendation assistant. You only answer 
questions about skincare concerns, ingredients, and product recommendations, 
using ONLY the data provided below. Do not use outside knowledge. Cite the 
ingredient/product name for every claim. If the provided data doesn't clearly 
answer the question, say you don't have enough information."""

def generate_response(query, retrieved_df):
    context = "\n".join(
        f"- {row.name}: {row.short_description} (benefits: {row.what_does_it_do})"
        for _, row in retrieved_df.iterrows()
    )
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"User question: {query}\n\nRelevant data:\n{context}"}
    ]
    completion = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=messages,
        temperature=0.2,
    )
    return completion.choices[0].message.content

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


In [25]:
def chatbot_reply(query):
    retrieved, max_sim = retrieve(query, model, embeddings, df)
    if retrieved is None:
        return "This is not my area of expertise — I can help with skincare concerns and product recommendations."
    return generate_response(query, retrieved)

In [ ]:
while True:
    q = input("You: ")
    if q.lower() in ("exit", "quit"):
        break
    pr
    int("Bot:", chatbot_reply(q))

Bot: This is not my area of expertise — I can help with skincare concerns and product recommendations.
